In [1]:
import pandas as pd

matches = pd.read_csv("D:/Powerbi/WPL_ANALYSIS/WPL_Performance_Analytics/data/matches.csv")
deliveries = pd.read_csv("D:/Powerbi/WPL_ANALYSIS/WPL_Performance_Analytics/data/deliveries.csv")

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)

print(matches.head())
print(deliveries.head())

Matches shape: (88, 22)
Deliveries shape: (20656, 26)
   match_id        date  season_label cricsheet_season  \
0   1358929  2023-03-04          2023          2022/23   
1   1358930  2023-03-05          2023          2022/23   
2   1358931  2023-03-05          2023          2022/23   
3   1358932  2023-03-06          2023          2022/23   
4   1358933  2023-03-07          2023          2022/23   

               event_name  match_number                               venue  \
0  Women's Premier League           1.0  Dr DY Patil Sports Academy, Mumbai   
1  Women's Premier League           2.0           Brabourne Stadium, Mumbai   
2  Women's Premier League           3.0  Dr DY Patil Sports Academy, Mumbai   
3  Women's Premier League           4.0           Brabourne Stadium, Mumbai   
4  Women's Premier League           5.0  Dr DY Patil Sports Academy, Mumbai   

          city                        team1                        team2  ...  \
0  Navi Mumbai               Mumbai India

In [2]:

# 1. Check missing values
print("MATCHES - Missing Values")
print(matches.isnull().sum()[matches.isnull().sum() > 0])

print("\nDELIVERIES - Missing Values")
print(deliveries.isnull().sum()[deliveries.isnull().sum() > 0])

# 2. Check duplicate records
print("Duplicate match IDs:", matches["match_id"].duplicated().sum())

print("Duplicate delivery rows:",
      deliveries.duplicated().sum())

# 3. Check season-wise match count
print("\nMatches per Season")
print(
    matches.groupby("season_label")["match_id"]
    .nunique()
    .reset_index(name="total_matches")
)

# 4. Check match date range
print("\nFirst match:", matches["date"].min())
print("Last match:", matches["date"].max())

# 5. Check unique teams
print("\nTeams:", sorted(
    set(matches["team1"].dropna()) |
    set(matches["team2"].dropna())
))


MATCHES - Missing Values
match_number       8
winner             1
win_by_runs       53
win_by_wickets    36
dtype: int64

DELIVERIES - Missing Values
wicket_kinds         19577
players_dismissed    19577
fielders_involved    19923
wicket_details       19577
dtype: int64
Duplicate match IDs: 0
Duplicate delivery rows: 0

Matches per Season
   season_label  total_matches
0          2023             22
1          2024             22
2          2025             22
3          2026             22

First match: 2023-03-04
Last match: 2026-02-05

Teams: ['Delhi Capitals', 'Gujarat Giants', 'Mumbai Indians', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'UP Warriorz']


In [3]:

# Check which team names exist before cleaning
print("Before cleaning:")
print(sorted(
    set(matches["team1"].dropna()) |
    set(matches["team2"].dropna())
))

# Standardize RCB name in both tables
team_name_mapping = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru"
}

for col in ["team1", "team2", "winner", "toss_winner"]:
    if col in matches.columns:
        matches[col] = matches[col].replace(team_name_mapping)

if "batting_team" in deliveries.columns:
    deliveries["batting_team"] = deliveries["batting_team"].replace(
        team_name_mapping
    )

if "bowling_team" in deliveries.columns:
    deliveries["bowling_team"] = deliveries["bowling_team"].replace(
        team_name_mapping
    )

# Verify the result
all_teams = (
    set(matches["team1"].dropna()) |
    set(matches["team2"].dropna())
)

print("\nAfter cleaning:")
print(sorted(all_teams))
print("\nTotal unique teams:", len(all_teams))

Before cleaning:
['Delhi Capitals', 'Gujarat Giants', 'Mumbai Indians', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'UP Warriorz']

After cleaning:
['Delhi Capitals', 'Gujarat Giants', 'Mumbai Indians', 'Royal Challengers Bengaluru', 'UP Warriorz']

Total unique teams: 5


In [4]:

# Matches with missing match_number
print("Matches with missing match_number:")

print(
    matches.loc[
        matches["match_number"].isna(),
        ["match_id", "date", "season_label",
         "team1", "team2", "winner", "result_type",
         "match_number"]
    ]
)

# Match with missing winner
print("\nMatches with missing winner:")

print(
    matches.loc[
        matches["winner"].isna(),
        ["match_id", "date", "season_label",
         "team1", "team2", "result_type",
         "win_by_runs", "win_by_wickets"]
    ]
)


Matches with missing match_number:
    match_id        date  season_label                        team1  \
20   1358949  2023-03-24          2023               Mumbai Indians   
21   1358950  2023-03-26          2023               Delhi Capitals   
42   1417736  2024-03-15          2024  Royal Challengers Bengaluru   
43   1417737  2024-03-17          2024               Delhi Capitals   
64   1469318  2025-03-13          2025               Mumbai Indians   
65   1469319  2025-03-15          2025               Mumbai Indians   
86   1513702  2026-02-03          2026               Gujarat Giants   
87   1513703  2026-02-05          2026               Delhi Capitals   

                          team2                       winner result_type  \
20                  UP Warriorz               Mumbai Indians         Win   
21               Mumbai Indians               Mumbai Indians         Win   
42               Mumbai Indians  Royal Challengers Bengaluru         Win   
43  Royal Challengers

In [5]:

# Count matches by result type
print("Match result types:")
print(
    matches["result_type"]
    .value_counts(dropna=False)
    .reset_index(name="match_count")
)

# Check all matches without a winner
print("\nMatches without a winner:")
print(
    matches.loc[
        matches["winner"].isna(),
        ["date", "season_label", "team1", "team2",
         "result_type", "winner"]
    ]
)

# Check result types season-wise
print("\nSeason-wise result types:")
print(
    matches.groupby(
        ["season_label", "result_type"],
        dropna=False
    ).size().reset_index(name="match_count")
)


Match result types:
  index  match_count
0   Win           87
1   tie            1

Matches without a winner:
          date  season_label                        team1        team2  \
52  2025-02-24          2025  Royal Challengers Bengaluru  UP Warriorz   

   result_type winner  
52         tie    NaN  

Season-wise result types:
   season_label result_type  match_count
0          2023         Win           22
1          2024         Win           22
2          2025         Win           21
3          2025         tie            1
4          2026         Win           22


In [6]:

# Inspect delivery data columns
print("Delivery columns:")
print(deliveries.columns.tolist())

# Check run-related columns
run_columns = [
    col for col in deliveries.columns
    if "run" in col.lower() or "extra" in col.lower()
]
print("\nRun and extra-related columns:")
print(run_columns)

# Show sample delivery records
print(
    deliveries[
        ["match_id", "date", "season_label", "innings_no",
         "batting_team", "batter", "bowler",
         "batter_runs", "total_runs"]
    ].head(10)
)


Delivery columns:
['match_id', 'date', 'season_label', 'cricsheet_season', 'innings_no', 'batting_team', 'over_number_0_based', 'delivery_in_over_index', 'actual_delivery', 'batter', 'non_striker', 'bowler', 'batter_runs', 'extras_runs', 'total_runs', 'byes', 'legbyes', 'wides', 'noballs', 'penalty_runs', 'wicket_count', 'wicket_kinds', 'players_dismissed', 'fielders_involved', 'wicket_details', 'source_file']

Run and extra-related columns:
['batter_runs', 'extras_runs', 'total_runs', 'penalty_runs']
   match_id        date  season_label  innings_no    batting_team  \
0   1358929  2023-03-04          2023           1  Mumbai Indians   
1   1358929  2023-03-04          2023           1  Mumbai Indians   
2   1358929  2023-03-04          2023           1  Mumbai Indians   
3   1358929  2023-03-04          2023           1  Mumbai Indians   
4   1358929  2023-03-04          2023           1  Mumbai Indians   
5   1358929  2023-03-04          2023           1  Mumbai Indians   
6   135892

In [7]:

# Create a working copy
batting_data = deliveries.copy()

# Mark boundary runs
batting_data["fours"] = (
    batting_data["batter_runs"] == 4
).astype(int)

batting_data["sixes"] = (
    batting_data["batter_runs"] == 6
).astype(int)

# Count balls faced:
# Wides are not counted as balls faced.
batting_data["balls_faced"] = (
    batting_data["wides"].fillna(0) == 0
).astype(int)

# Player batting summary
batting_summary = (
    batting_data.groupby("batter")
    .agg(
        total_runs=("batter_runs", "sum"),
        balls_faced=("balls_faced", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum")
    )
    .reset_index()
)

# Strike rate
batting_summary["strike_rate"] = (
    batting_summary["total_runs"] /
    batting_summary["balls_faced"].replace(0, pd.NA)
) * 100

# Sort highest run scorers first
batting_summary = batting_summary.sort_values(
    "total_runs", ascending=False
)

print(batting_summary.head(10).round(2))


              batter  total_runs  balls_faced  fours  sixes  strike_rate
69   NR Sciver-Brunt        1348          936    205     15       144.02
64        MM Lanning        1200          947    179     16       126.72
38            H Kaur        1193          820    148     35       145.49
111    Shafali Verma        1124          739    132     53       152.10
93        S Mandhana        1025          751    145     29       136.48
30          EA Perry         972          731    109     25       132.97
40       HK Matthews         867          711    122     24       121.94
36         GM Harris         819          548    112     28       149.45
82          RM Ghosh         816          542     88     41       150.55
1          A Gardner         811          572     83     34       141.78


In [8]:

# 1. Validate total batter runs
print("Total batter runs:", deliveries["batter_runs"].sum())

# 2. Check wide and no-ball records
print("Wide deliveries:", (deliveries["wides"].fillna(0) > 0).sum())
print("No-ball deliveries:", (deliveries["noballs"].fillna(0) > 0).sum())

# 3. Inspect dismissal records
dismissals = deliveries.loc[
    deliveries["players_dismissed"].notna(),
    ["match_id", "date", "season_label", "batter",
     "wicket_kinds", "players_dismissed"]
]

print("\nDismissal records:", len(dismissals))
print(dismissals.head(15))

# 4. Check whether the boundary counts reconcile
print("\nBoundary runs from fours:", batting_summary["fours"].sum() * 4)
print("Boundary runs from sixes:", batting_summary["sixes"].sum() * 6)


Total batter runs: 26081
Wide deliveries: 592
No-ball deliveries: 64

Dismissal records: 1079
     match_id        date  season_label           batter  wicket_kinds  \
15    1358929  2023-03-04          2023        YH Bhatia        caught   
55    1358929  2023-03-04          2023  NR Sciver-Brunt        caught   
62    1358929  2023-03-04          2023      HK Matthews        bowled   
105   1358929  2023-03-04          2023           H Kaur        caught   
122   1358929  2023-03-04          2023      P Vastrakar        caught   
127   1358929  2023-03-04          2023        BL Mooney  retired hurt   
129   1358929  2023-03-04          2023           H Deol        caught   
132   1358929  2023-03-04          2023        A Gardner        caught   
139   1358929  2023-03-04          2023        S Meghana        bowled   
152   1358929  2023-03-04          2023     A Sutherland        bowled   
165   1358929  2023-03-04          2023        G Wareham        bowled   
167   1358929  202

In [9]:

# Count wicket types
print("Wicket types:")
print(
    deliveries["wicket_kinds"]
    .dropna()
    .value_counts()
    .reset_index(name="count")
)

# Check whether multiple players are dismissed on one delivery
print("\nSample records with dismissal details:")
print(
    deliveries.loc[
        deliveries["players_dismissed"].notna(),
        ["date", "season_label", "batter", "wicket_kinds",
         "players_dismissed"]
    ].tail(20)
)


Wicket types:
               index  count
0             caught    589
1             bowled    219
2                lbw     88
3            run out     76
4            stumped     68
5  caught and bowled     36
6        retired out      2
7       retired hurt      1

Sample records with dismissal details:
             date  season_label         batter       wicket_kinds  \
20162  2026-02-01          2026         M Kapp             caught   
20168  2026-02-01          2026       CA Henry  caught and bowled   
20195  2026-02-03          2026     SFM Devine             caught   
20215  2026-02-03          2026      AB Sharma             caught   
20216  2026-02-03          2026      A Gardner             caught   
20238  2026-02-03          2026       KS Ahuja            stumped   
20285  2026-02-03          2026      G Wareham             caught   
20288  2026-02-03          2026     BS Fulmali  caught and bowled   
20309  2026-02-03          2026      BL Mooney            run out   
2035

In [10]:

# Count wickets using the wicket_count column
print("Total wicket events:", deliveries["wicket_count"].sum())

# Count deliveries that contain wicket records
print(
    "Deliveries with wicket records:",
    deliveries["wicket_count"].gt(0).sum()
)

# Check how many deliveries contain more than one wicket
print(
    "Deliveries with multiple wickets:",
    deliveries["wicket_count"].gt(1).sum()
)

# Check whether any wicket records have missing dismissal details
wicket_rows = deliveries[deliveries["wicket_count"] > 0]

print(
    "Wicket rows with missing wicket_kinds:",
    wicket_rows["wicket_kinds"].isna().sum()
)


Total wicket events: 1079
Deliveries with wicket records: 1079
Deliveries with multiple wickets: 0
Wicket rows with missing wicket_kinds: 0


In [11]:

# Make a copy so our original deliveries DataFrame stays unchanged
bowling_data = deliveries.copy()

# Wicket types credited to the bowler
bowler_credit_wickets = [
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped"
]

# Create a flag for wickets credited to the bowler
bowling_data["bowler_wicket"] = (
    bowling_data["wicket_kinds"]
    .isin(bowler_credit_wickets)
    .astype(int)
)

# Runs conceded by the bowler:
# Exclude byes, leg-byes and penalty runs.
# Wides and no-balls are included in runs conceded.
bowling_data["bowler_runs_conceded"] = (
    bowling_data["total_runs"]
    - bowling_data["byes"].fillna(0)
    - bowling_data["legbyes"].fillna(0)
    - bowling_data["penalty_runs"].fillna(0)
)

# Legal balls: wides and no-balls do not count as legal balls
bowling_data["legal_ball"] = (
    (bowling_data["wides"].fillna(0) == 0)
    & (bowling_data["noballs"].fillna(0) == 0)
).astype(int)

# Aggregate bowling statistics by player
bowling_summary = (
    bowling_data.groupby("bowler")
    .agg(
        wickets=("bowler_wicket", "sum"),
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum")
    )
    .reset_index()
)

# Calculate overs and economy rate
bowling_summary["overs"] = (
    bowling_summary["legal_balls"] // 6
    + (bowling_summary["legal_balls"] % 6) / 10
)

bowling_summary["economy"] = (
    bowling_summary["runs_conceded"]
    / (bowling_summary["legal_balls"] / 6).replace(0, pd.NA)
)

# Show top 10 wicket-takers
bowling_summary = bowling_summary.sort_values(
    ["wickets", "economy"],
    ascending=[False, True]
)

print(bowling_summary.head(10).round(2))


             bowler  wickets  runs_conceded  legal_balls  overs  economy
7           AC Kerr       54            926          730  121.4     7.61
27      HK Matthews       46            863          680  113.2     7.61
68     S Ecclestone       43            877          768  128.0     6.85
53  NR Sciver-Brunt       40            960          714  119.0     8.07
44           M Kapp       38            804          798  133.0     6.05
74         S Pandey       37            936          780  130.0     7.20
13        DB Sharma       34           1011          722  120.2     8.40
30      JL Jonassen       33            685          531   88.3     7.74
1         A Gardner       31            931          639  106.3     8.74
83         SR Patil       30            624          429   71.3     8.73


In [12]:

# 1. Compare total wickets credited to bowlers with total wicket events
print("All recorded wicket events:", deliveries["wicket_count"].sum())
print("Wickets credited to bowlers:", bowling_data["bowler_wicket"].sum())

# 2. Check the types of wickets that are NOT credited to bowlers
non_bowler_wickets = deliveries[
    deliveries["wicket_kinds"].notna()
    & ~deliveries["wicket_kinds"].isin(bowler_credit_wickets)
]

print("\nWicket types not credited to bowlers:")
print(
    non_bowler_wickets["wicket_kinds"]
    .value_counts()
    .reset_index(name="count")
)

# 3. Check for any negative bowler-conceded runs
print(
    "\nRows with negative bowler-conceded runs:",
    (bowling_data["bowler_runs_conceded"] < 0).sum()
)

# 4. Check the total legal balls and conceded runs
print("\nTotal legal balls:", bowling_data["legal_ball"].sum())
print("Total bowler runs conceded:", bowling_data["bowler_runs_conceded"].sum())


All recorded wicket events: 1079
Wickets credited to bowlers: 1000

Wicket types not credited to bowlers:
          index  count
0       run out     76
1   retired out      2
2  retired hurt      1

Rows with negative bowler-conceded runs: 0

Total legal balls: 20000
Total bowler runs conceded: 26932


In [13]:

# Calculate bowling average safely
bowling_summary["bowling_average"] = (
    bowling_summary["runs_conceded"]
    / bowling_summary["wickets"].where(
        bowling_summary["wickets"] != 0
    )
)

# Sort by wickets, then economy
bowling_summary = bowling_summary.sort_values(
    ["wickets", "economy"],
    ascending=[False, True]
)

# Display top 10 bowlers
print(
    bowling_summary[
        [
            "bowler",
            "wickets",
            "runs_conceded",
            "legal_balls",
            "overs",
            "economy",
            "bowling_average"
        ]
    ].head(10).round(2)
)


             bowler  wickets  runs_conceded  legal_balls  overs  economy  \
7           AC Kerr       54            926          730  121.4     7.61   
27      HK Matthews       46            863          680  113.2     7.61   
68     S Ecclestone       43            877          768  128.0     6.85   
53  NR Sciver-Brunt       40            960          714  119.0     8.07   
44           M Kapp       38            804          798  133.0     6.05   
74         S Pandey       37            936          780  130.0     7.20   
13        DB Sharma       34           1011          722  120.2     8.40   
30      JL Jonassen       33            685          531   88.3     7.74   
1         A Gardner       31            931          639  106.3     8.74   
83         SR Patil       30            624          429   71.3     8.73   

    bowling_average  
7             17.15  
27            18.76  
68            20.40  
53            24.00  
44            21.16  
74            25.30  
13       

In [14]:

# Group bowling data by season and bowler
season_bowling = (
    bowling_data.groupby(["season_label", "bowler"])
    .agg(
        wickets=("bowler_wicket", "sum"),
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum")
    )
    .reset_index()
)

# Calculate economy rate
season_bowling["economy"] = (
    season_bowling["runs_conceded"]
    / (season_bowling["legal_balls"] / 6).where(
        season_bowling["legal_balls"] > 0
    )
)

# Calculate bowling average
season_bowling["bowling_average"] = (
    season_bowling["runs_conceded"]
    / season_bowling["wickets"].where(
        season_bowling["wickets"] > 0
    )
)

# Show top 5 wicket-takers in each season
top_bowlers_each_season = (
    season_bowling.sort_values(
        ["season_label", "wickets", "economy"],
        ascending=[True, False, True]
    )
    .groupby("season_label")
    .head(5)
)

print(top_bowlers_each_season.round(2))


     season_label           bowler  wickets  runs_conceded  legal_balls  \
15           2023      HK Matthews       16            202          204   
37           2023     S Ecclestone       16            235          213   
5            2023          AC Kerr       15            211          196   
16           2023        IECM Wong       15            210          195   
38           2023        S Ishaque       15            244          209   
93           2024         SR Patil       13            157          129   
80           2024           S Asha       12            185          156   
84           2024       S Molineux       12            278          228   
69           2024           M Kapp       11            176          168   
81           2024     S Ecclestone       11            207          189   
105          2025          AC Kerr       18            287          222   
119          2025      HK Matthews       18            307          224   
120          2025      JL

In [15]:

# Check dismissal types for each batter
dismissal_check = (
    deliveries[deliveries["players_dismissed"].notna()]
    .groupby(["players_dismissed", "wicket_kinds"])
    .size()
    .reset_index(name="count")
)

print(dismissal_check.head(20))

print(
    "Total dismissal records:",
    deliveries["players_dismissed"].notna().sum()
)


   players_dismissed       wicket_kinds  count
0           A Capsey             bowled      4
1           A Capsey             caught      8
2           A Capsey                lbw      1
3           A Capsey            run out      1
4           A Capsey            stumped      1
5          A Gardner             bowled      6
6          A Gardner             caught     17
7          A Gardner  caught and bowled      1
8          A Gardner                lbw      3
9          A Gardner            stumped      5
10           A Reddy             bowled      1
11           A Reddy             caught      7
12           A Reddy  caught and bowled      1
13           A Reddy                lbw      1
14            A Soni        retired out      1
15      A Sutherland             bowled      3
16      A Sutherland             caught      7
17      A Sutherland                lbw      1
18      A Sutherland            stumped      1
19           AB Kaur             bowled      5
Total dismiss

In [16]:

# Dismissal types that count as a batter being out
valid_dismissals = [
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "run out",
    "stumped",
    "retired out"
]

# Count dismissals against the actual dismissed player
dismissal_summary = (
    deliveries[
        deliveries["wicket_kinds"].isin(valid_dismissals)
        & deliveries["players_dismissed"].notna()
    ]
    .groupby("players_dismissed")
    .size()
    .reset_index(name="dismissals")
    .rename(columns={"players_dismissed": "batter"})
)

print(
    dismissal_summary.sort_values(
        "dismissals", ascending=False
    ).head(10)
)

print("Total dismissals counted:", dismissal_summary["dismissals"].sum())


              batter  dismissals
101    Shafali Verma          34
39       HK Matthews          33
83        S Mandhana          33
61        MM Lanning          32
1          A Gardner          32
49        KP Navgire          30
65   NR Sciver-Brunt          27
35         GM Harris          27
112        YH Bhatia          27
37            H Kaur          26
Total dismissals counted: 1078


In [17]:

# Merge dismissal counts into the batting summary
batting_summary = batting_summary.drop(
    columns=["dismissals"], errors="ignore"
)

batting_summary = batting_summary.merge(
    dismissal_summary,
    on="batter",
    how="left"
)

# Players without recorded dismissals get zero
batting_summary["dismissals"] = (
    batting_summary["dismissals"].fillna(0).astype(int)
)

# Calculate batting average
batting_summary["batting_average"] = (
    batting_summary["total_runs"]
    / batting_summary["dismissals"].where(
        batting_summary["dismissals"] > 0
    )
)

# Sort by total runs
batting_summary = batting_summary.sort_values(
    "total_runs", ascending=False
)

# Display top 10 batters
print(
    batting_summary[
        [
            "batter",
            "total_runs",
            "dismissals",
            "batting_average",
            "balls_faced",
            "strike_rate",
            "fours",
            "sixes"
        ]
    ].head(10).round(2)
)


            batter  total_runs  dismissals  batting_average  balls_faced  \
0  NR Sciver-Brunt        1348          27            49.93          936   
1       MM Lanning        1200          32            37.50          947   
2           H Kaur        1193          26            45.88          820   
3    Shafali Verma        1124          34            33.06          739   
4       S Mandhana        1025          33            31.06          751   
5         EA Perry         972          15            64.80          731   
6      HK Matthews         867          33            26.27          711   
7        GM Harris         819          27            30.33          548   
8         RM Ghosh         816          24            34.00          542   
9        A Gardner         811          32            25.34          572   

   strike_rate  fours  sixes  
0       144.02    205     15  
1       126.72    179     16  
2       145.49    148     35  
3       152.10    132     53  
4       

In [18]:

# Combine both team columns to find each team's matches played
team1_matches = matches[["match_id", "team1"]].rename(
    columns={"team1": "team"}
)

team2_matches = matches[["match_id", "team2"]].rename(
    columns={"team2": "team"}
)

team_matches = pd.concat(
    [team1_matches, team2_matches],
    ignore_index=True
)

# Matches played by each team
matches_played = (
    team_matches.groupby("team")["match_id"]
    .nunique()
    .reset_index(name="matches_played")
)

# Matches won by each team
matches_won = (
    matches[matches["winner"].notna()]
    .groupby("winner")["match_id"]
    .nunique()
    .reset_index(name="matches_won")
    .rename(columns={"winner": "team"})
)

# Merge results
team_summary = matches_played.merge(
    matches_won,
    on="team",
    how="left"
)

team_summary["matches_won"] = (
    team_summary["matches_won"].fillna(0).astype(int)
)

# Win percentage
team_summary["win_percentage"] = (
    team_summary["matches_won"]
    / team_summary["matches_played"]
) * 100

team_summary = team_summary.sort_values(
    "matches_won", ascending=False
)

print(team_summary.round(2))


                          team  matches_played  matches_won  win_percentage
2               Mumbai Indians              37           23           62.16
0               Delhi Capitals              37           22           59.46
3  Royal Challengers Bengaluru              35           18           51.43
1               Gujarat Giants              34           13           38.24
4                  UP Warriorz              33           11           33.33


In [19]:

# Season-wise matches played by each team
season_team_matches = pd.concat(
    [
        matches[["season_label", "team1"]].rename(
            columns={"team1": "team"}
        ),
        matches[["season_label", "team2"]].rename(
            columns={"team2": "team"}
        )
    ],
    ignore_index=True
)

season_matches_played = (
    season_team_matches.groupby(["season_label", "team"])["season_label"]
    .size()
    .reset_index(name="matches_played")
)

# Season-wise matches won
season_matches_won = (
    matches[matches["winner"].notna()]
    .groupby(["season_label", "winner"])["match_id"]
    .nunique()
    .reset_index(name="matches_won")
    .rename(columns={"winner": "team"})
)

# Combine played and won
season_team_summary = season_matches_played.merge(
    season_matches_won,
    on=["season_label", "team"],
    how="left"
)

season_team_summary["matches_won"] = (
    season_team_summary["matches_won"].fillna(0).astype(int)
)

season_team_summary["win_percentage"] = (
    season_team_summary["matches_won"]
    / season_team_summary["matches_played"]
) * 100

season_team_summary = season_team_summary.sort_values(
    ["season_label", "matches_won"],
    ascending=[True, False]
)

print(season_team_summary.round(2))


    season_label                         team  matches_played  matches_won  \
2           2023               Mumbai Indians              10            8   
0           2023               Delhi Capitals               9            6   
4           2023                  UP Warriorz               9            4   
1           2023               Gujarat Giants               8            2   
3           2023  Royal Challengers Bengaluru               8            2   
5           2024               Delhi Capitals               9            6   
8           2024  Royal Challengers Bengaluru              10            6   
7           2024               Mumbai Indians               9            5   
9           2024                  UP Warriorz               8            3   
6           2024               Gujarat Giants               8            2   
12          2025               Mumbai Indians              10            7   
10          2025               Delhi Capitals               9   

In [20]:

# Total runs scored by each batting team
team_runs = (
    deliveries.groupby("batting_team")
    .agg(
        total_runs=("total_runs", "sum"),
        batter_runs=("batter_runs", "sum"),
        innings_count=("innings_no", "nunique")
    )
    .reset_index()
    .rename(columns={"batting_team": "team"})
)

team_runs = team_runs.sort_values(
    "total_runs", ascending=False
)

print(team_runs.round(2))


                          team  total_runs  batter_runs  innings_count
2               Mumbai Indians        5816         5568              2
0               Delhi Capitals        5672         5417              2
3  Royal Challengers Bengaluru        5594         5353              3
1               Gujarat Giants        5294         5022              2
4                  UP Warriorz        5002         4721              3


In [21]:

# Count actual innings played by each batting team
team_innings = (
    deliveries[["match_id", "innings_no", "batting_team"]]
    .drop_duplicates()
    .groupby("batting_team")
    .size()
    .reset_index(name="innings_played")
    .rename(columns={"batting_team": "team"})
)

# Remove the old incorrect innings_count column
team_runs = team_runs.drop(
    columns=["innings_count"], errors="ignore"
)

# Merge actual innings count
team_runs = team_runs.merge(
    team_innings,
    on="team",
    how="left"
)

# Calculate average runs per innings
team_runs["runs_per_innings"] = (
    team_runs["total_runs"] / team_runs["innings_played"]
)

# Sort and display
team_runs = team_runs.sort_values(
    "total_runs", ascending=False
)

print(team_runs.round(2))


                          team  total_runs  batter_runs  innings_played  \
0               Mumbai Indians        5816         5568              37   
1               Delhi Capitals        5672         5417              37   
2  Royal Challengers Bengaluru        5594         5353              36   
3               Gujarat Giants        5294         5022              34   
4                  UP Warriorz        5002         4721              34   

   runs_per_innings  
0            157.19  
1            153.30  
2            155.39  
3            155.71  
4            147.12  


In [22]:

# Check whether bowling_team already exists
print(deliveries.columns.tolist())


['match_id', 'date', 'season_label', 'cricsheet_season', 'innings_no', 'batting_team', 'over_number_0_based', 'delivery_in_over_index', 'actual_delivery', 'batter', 'non_striker', 'bowler', 'batter_runs', 'extras_runs', 'total_runs', 'byes', 'legbyes', 'wides', 'noballs', 'penalty_runs', 'wicket_count', 'wicket_kinds', 'players_dismissed', 'fielders_involved', 'wicket_details', 'source_file']


In [23]:

# Get both teams from the matches dataset
match_teams = matches[["match_id", "team1", "team2"]].copy()

# Add team names to each delivery row
deliveries = deliveries.merge(
    match_teams,
    on="match_id",
    how="left"
)

# Identify the bowling team
deliveries["bowling_team"] = deliveries.apply(
    lambda row: (
        row["team2"]
        if row["batting_team"] == row["team1"]
        else row["team1"]
    ),
    axis=1
)

# Check the result
print(
    deliveries[
        ["match_id", "innings_no", "batting_team", "bowling_team"]
    ].head(10)
)


   match_id  innings_no    batting_team    bowling_team
0   1358929           1  Mumbai Indians  Gujarat Giants
1   1358929           1  Mumbai Indians  Gujarat Giants
2   1358929           1  Mumbai Indians  Gujarat Giants
3   1358929           1  Mumbai Indians  Gujarat Giants
4   1358929           1  Mumbai Indians  Gujarat Giants
5   1358929           1  Mumbai Indians  Gujarat Giants
6   1358929           1  Mumbai Indians  Gujarat Giants
7   1358929           1  Mumbai Indians  Gujarat Giants
8   1358929           1  Mumbai Indians  Gujarat Giants
9   1358929           1  Mumbai Indians  Gujarat Giants


In [24]:

# Check for missing team mappings
print("Missing bowling teams:", deliveries["bowling_team"].isna().sum())

# Check that batting and bowling teams are different
print(
    "Same batting and bowling team:",
    (deliveries["batting_team"] == deliveries["bowling_team"]).sum()
)


Missing bowling teams: 0
Same batting and bowling team: 0


In [25]:

# Count only wickets credited to the bowler
bowler_credit_wickets = [
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped"
]

# Create bowling analysis data
bowling_data = deliveries.copy()

bowling_data["bowler_wicket"] = (
    bowling_data["wicket_kinds"]
    .isin(bowler_credit_wickets)
    .astype(int)
)

# Runs conceded by the bowling team
bowling_data["bowler_runs_conceded"] = (
    bowling_data["total_runs"]
    - bowling_data["byes"].fillna(0)
    - bowling_data["legbyes"].fillna(0)
    - bowling_data["penalty_runs"].fillna(0)
)

# Legal deliveries: wides and no-balls are not legal balls
bowling_data["legal_ball"] = (
    (bowling_data["wides"].fillna(0) == 0)
    & (bowling_data["noballs"].fillna(0) == 0)
).astype(int)

# Summarize by bowling team
team_bowling = (
    bowling_data.groupby("bowling_team")
    .agg(
        wickets=("bowler_wicket", "sum"),
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum")
    )
    .reset_index()
    .rename(columns={"bowling_team": "team"})
)

# Calculate overs and economy rate
team_bowling["overs"] = (
    team_bowling["legal_balls"] // 6
    + (team_bowling["legal_balls"] % 6) / 10
)

team_bowling["economy"] = (
    team_bowling["runs_conceded"]
    / (team_bowling["legal_balls"] / 6)
).round(2)

# Best economy first
team_bowling = team_bowling.sort_values(
    "economy", ascending=True
)

print(team_bowling)


                          team  wickets  runs_conceded  legal_balls  overs  \
0               Delhi Capitals      229           5476         4362  727.0   
2               Mumbai Indians      240           5335         4197  699.3   
3  Royal Challengers Bengaluru      196           5531         3975  662.3   
4                  UP Warriorz      158           5213         3747  624.3   
1               Gujarat Giants      177           5377         3719  619.5   

   economy  
0     7.53  
2     7.63  
3     8.35  
4     8.35  
1     8.67  


In [26]:

# Combine batting and bowling summaries
team_performance = team_runs.merge(
    team_bowling,
    on="team",
    how="inner"
)

# Select useful columns
team_performance = team_performance[
    [
        "team",
        "total_runs",
        "runs_per_innings",
        "wickets",
        "runs_conceded",
        "economy"
    ]
]

print(
    team_performance.sort_values(
        "total_runs",
        ascending=False
    ).round(2)
)


                          team  total_runs  runs_per_innings  wickets  \
0               Mumbai Indians        5816            157.19      240   
1               Delhi Capitals        5672            153.30      229   
2  Royal Challengers Bengaluru        5594            155.39      196   
3               Gujarat Giants        5294            155.71      177   
4                  UP Warriorz        5002            147.12      158   

   runs_conceded  economy  
0           5335     7.63  
1           5476     7.53  
2           5531     8.35  
3           5377     8.67  
4           5213     8.35  


In [27]:

# Season-wise team batting performance
season_batting = (
    deliveries.groupby(["season_label", "batting_team"])
    .agg(
        total_runs=("total_runs", "sum"),
        batter_runs=("batter_runs", "sum")
    )
    .reset_index()
    .rename(columns={"batting_team": "team"})
)

# Count actual batting innings for each team and season
season_innings = (
    deliveries[
        ["season_label", "match_id", "innings_no", "batting_team"]
    ]
    .drop_duplicates()
    .groupby(["season_label", "batting_team"])
    .size()
    .reset_index(name="innings_played")
    .rename(columns={"batting_team": "team"})
)

# Merge and calculate runs per innings
season_batting = season_batting.merge(
    season_innings,
    on=["season_label", "team"],
    how="left"
)

season_batting["runs_per_innings"] = (
    season_batting["total_runs"]
    / season_batting["innings_played"]
)

season_batting = season_batting.sort_values(
    ["season_label", "total_runs"],
    ascending=[True, False]
)

print(season_batting.round(2))


    season_label                         team  total_runs  batter_runs  \
2           2023               Mumbai Indians        1482         1420   
4           2023                  UP Warriorz        1335         1263   
0           2023               Delhi Capitals        1319         1246   
3           2023  Royal Challengers Bengaluru        1246         1197   
1           2023               Gujarat Giants        1159         1126   
8           2024  Royal Challengers Bengaluru        1490         1433   
5           2024               Delhi Capitals        1403         1357   
7           2024               Mumbai Indians        1353         1291   
6           2024               Gujarat Giants        1180         1102   
9           2024                  UP Warriorz        1155         1087   
12          2025               Mumbai Indians        1604         1532   
11          2025               Gujarat Giants        1418         1346   
13          2025  Royal Challengers Be

In [28]:

season_trend = season_batting.pivot(
    index="season_label",
    columns="team",
    values="runs_per_innings"
)

print(season_trend.round(2))


team          Delhi Capitals  Gujarat Giants  Mumbai Indians  \
season_label                                                   
2023                  146.56          144.88          148.20   
2024                  155.89          147.50          150.33   
2025                  149.00          157.56          160.40   
2026                  160.90          170.78          172.12   

team          Royal Challengers Bengaluru  UP Warriorz  
season_label                                            
2023                               155.75       148.33  
2024                               149.00       144.38  
2025                               153.67       144.00  
2026                               163.89       152.00  


In [29]:

# Season-wise bowling performance
season_team_bowling = (
    bowling_data.groupby(["season_label", "bowling_team"])
    .agg(
        wickets=("bowler_wicket", "sum"),
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum")
    )
    .reset_index()
    .rename(columns={"bowling_team": "team"})
)

# Calculate economy rate
season_team_bowling["economy"] = (
    season_team_bowling["runs_conceded"]
    / (season_team_bowling["legal_balls"] / 6)
).round(2)

# Count actual matches for each team per season
season_match_count = (
    season_team_matches
    .groupby(["season_label", "team"])["team"]
    .size()
    .reset_index(name="matches_played")
)

# Add matches played to bowling data
season_team_bowling = season_team_bowling.merge(
    season_match_count,
    on=["season_label", "team"],
    how="left"
)

# Wickets per match
season_team_bowling["wickets_per_match"] = (
    season_team_bowling["wickets"]
    / season_team_bowling["matches_played"]
).round(2)

print(
    season_team_bowling.sort_values(
        ["season_label", "economy"],
        ascending=[True, True]
    ).round(2)
)


    season_label                         team  wickets  runs_conceded  \
2           2023               Mumbai Indians       74           1174   
0           2023               Delhi Capitals       47           1210   
4           2023                  UP Warriorz       47           1425   
3           2023  Royal Challengers Bengaluru       32           1306   
1           2023               Gujarat Giants       40           1322   
5           2024               Delhi Capitals       60           1297   
7           2024               Mumbai Indians       51           1316   
8           2024  Royal Challengers Bengaluru       61           1483   
9           2024                  UP Warriorz       40           1215   
6           2024               Gujarat Giants       32           1158   
10          2025               Delhi Capitals       57           1340   
12          2025               Mumbai Indians       69           1524   
11          2025               Gujarat Giants      

In [30]:

# Check bowling data totals by season
season_bowling_check = (
    season_team_bowling.groupby("season_label")
    .agg(
        total_bowler_wickets=("wickets", "sum"),
        total_runs_conceded=("runs_conceded", "sum"),
        total_legal_balls=("legal_balls", "sum")
    )
    .reset_index()
)

season_bowling_check["overall_economy"] = (
    season_bowling_check["total_runs_conceded"]
    / (season_bowling_check["total_legal_balls"] / 6)
).round(2)

print(season_bowling_check)


   season_label  total_bowler_wickets  total_runs_conceded  total_legal_balls  \
0          2023                   240                 6437               4856   
1          2024                   244                 6469               5024   
2          2025                   262                 6933               5049   
3          2026                   254                 7093               5071   

   overall_economy  
0             7.95  
1             7.73  
2             8.24  
3             8.39  


In [31]:
from pathlib import Path

# Create a folder for Power BI files
output_folder = Path("D:/Powerbi/WPL_ANALYSIS")
output_folder.mkdir(parents=True, exist_ok=True)

# Export analysis tables
export_tables = {
    "matches_cleaned.csv": matches,
    "team_performance.csv": team_performance,
    "season_batting.csv": season_batting,
    "season_team_bowling.csv": season_team_bowling,
    "batting_summary.csv": batting_summary,
    "bowling_summary.csv": bowling_summary
}

for filename, dataframe in export_tables.items():
    dataframe.to_csv(
        output_folder / filename,
        index=False
    )
    print(f"Exported: {filename} | Rows: {len(dataframe)}")

print("CSV export completed!")


Exported: matches_cleaned.csv | Rows: 88
Exported: team_performance.csv | Rows: 5
Exported: season_batting.csv | Rows: 20
Exported: season_team_bowling.csv | Rows: 20
Exported: batting_summary.csv | Rows: 123
Exported: bowling_summary.csv | Rows: 96
CSV export completed!


In [32]:

# Player batting performance by season
season_player_batting = (
    deliveries.groupby(["season_label", "batter"])
    .agg(
        runs=("batter_runs", "sum"),
        fours=("batter_runs", lambda x: (x == 4).sum()),
        sixes=("batter_runs", lambda x: (x == 6).sum()),
        balls_faced=("wides", lambda x: (x.fillna(0) == 0).sum())
    )
    .reset_index()
)

season_player_batting["strike_rate"] = (
    season_player_batting["runs"]
    / season_player_batting["balls_faced"].where(
        season_player_batting["balls_faced"] > 0
    )
    * 100
)

# Highest run scorer in each season
best_batter_each_season = (
    season_player_batting.sort_values(
        ["season_label", "runs", "strike_rate"],
        ascending=[True, False, False]
    )
    .groupby("season_label")
    .head(1)
    .reset_index(drop=True)
)

print(best_batter_each_season.round(2))


   season_label           batter  runs  fours  sixes  balls_faced  strike_rate
0          2023       MM Lanning   345     50      6          248       139.11
1          2024         EA Perry   347     41      7          276       125.72
2          2025  NR Sciver-Brunt   523     84      4          343       152.48
3          2026       S Mandhana   377     57      9          246       153.25


In [33]:

# Player bowling performance by season
season_player_bowling = (
    bowling_data.groupby(["season_label", "bowler"])
    .agg(
        wickets=("bowler_wicket", "sum"),
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum")
    )
    .reset_index()
)

season_player_bowling["economy"] = (
    season_player_bowling["runs_conceded"]
    / (season_player_bowling["legal_balls"] / 6).where(
        season_player_bowling["legal_balls"] > 0
    )
)

# Highest wicket taker in each season
best_bowler_each_season = (
    season_player_bowling.sort_values(
        ["season_label", "wickets", "economy"],
        ascending=[True, False, True]
    )
    .groupby("season_label")
    .head(1)
    .reset_index(drop=True)
)

print(best_bowler_each_season.round(2))


   season_label       bowler  wickets  runs_conceded  legal_balls  economy
0          2023  HK Matthews       16            202          204     5.94
1          2024     SR Patil       13            157          129     7.30
2          2025      AC Kerr       18            287          222     7.76
3          2026   SFM Devine       17            272          197     8.28


In [34]:
from pathlib import Path

output_folder = Path("D:/Powerbi/WPL_ANALYSIS")
output_folder.mkdir(parents=True, exist_ok=True)

best_batter_each_season.to_csv(
    output_folder / "best_batter_each_season.csv",
    index=False
)

best_bowler_each_season.to_csv(
    output_folder / "best_bowler_each_season.csv",
    index=False
)

print("Best batter and bowler tables exported successfully!")

Best batter and bowler tables exported successfully!


In [35]:
# Dismissal types that count towards batting average
dismissal_types = [
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "run out",
    "stumped",
    "retired out"
]

# Keep only actual dismissals with a batter dismissed
season_dismissals = deliveries[
    deliveries["wicket_kinds"].isin(dismissal_types)
    & deliveries["players_dismissed"].notna()
].copy()

# Count dismissals by season and player
season_dismissal_summary = (
    season_dismissals
    .groupby(["season_label", "players_dismissed"])
    .size()
    .reset_index(name="dismissals")
    .rename(columns={"players_dismissed": "batter"})
)

print(season_dismissal_summary.head(10))

   season_label          batter  dismissals
0          2023        A Capsey           6
1          2023       A Gardner           7
2          2023         A Reddy           2
3          2023    A Sutherland           4
4          2023         AB Kaur           3
5          2023         AC Kerr           4
6          2023        AJ Healy           8
7          2023  Ashwani Kumari           1
8          2023     D Hemalatha           5
9          2023       DB Sharma           7


In [36]:
# Merge season-wise dismissals with batting stats
season_player_batting_avg = season_player_batting.merge(
    season_dismissal_summary,
    on=["season_label", "batter"],
    how="left"
)

# Players with no dismissals get 0
season_player_batting_avg["dismissals"] = (
    season_player_batting_avg["dismissals"].fillna(0)
)

# Calculate batting average
season_player_batting_avg["batting_average"] = (
    season_player_batting_avg["runs"]
    / season_player_batting_avg["dismissals"].where(
        season_player_batting_avg["dismissals"] > 0
    )
)

print(
    season_player_batting_avg[
        ["season_label", "batter", "runs", "dismissals", "batting_average"]
    ].head(10).round(2)
)

   season_label          batter  runs  dismissals  batting_average
0          2023        A Capsey   159         6.0            26.50
1          2023       A Gardner   204         7.0            29.14
2          2023         A Reddy    25         2.0            12.50
3          2023    A Sutherland    28         4.0             7.00
4          2023         AB Kaur    24         3.0             8.00
5          2023         AC Kerr   149         4.0            37.25
6          2023        AJ Healy   253         8.0            31.62
7          2023  Ashwani Kumari     5         1.0             5.00
8          2023       BL Mooney     0         0.0              NaN
9          2023     D Hemalatha   151         5.0            30.20


In [37]:
# Add batting average to the season-wise best batter table
best_batter_each_season = best_batter_each_season.merge(
    season_player_batting_avg[
        ["season_label", "batter", "batting_average"]
    ],
    on=["season_label", "batter"],
    how="left"
)

print(best_batter_each_season.round(2))

   season_label           batter  runs  fours  sixes  balls_faced  \
0          2023       MM Lanning   345     50      6          248   
1          2024         EA Perry   347     41      7          276   
2          2025  NR Sciver-Brunt   523     84      4          343   
3          2026       S Mandhana   377     57      9          246   

   strike_rate  batting_average  
0       139.11            49.29  
1       125.72            69.40  
2       152.48            65.38  
3       153.25            53.86  


In [38]:
best_batter_each_season.to_csv(
    output_folder / "best_batter_each_season.csv",
    index=False
)

print("Best batter table updated with batting average!")

Best batter table updated with batting average!


In [39]:

# Create player image lookup CSV files

batting_images = (
    batting_summary[["batter"]]
    .drop_duplicates()
    .sort_values("batter")
    .copy()
)
batting_images["photo_url"] = ""

bowling_images = (
    bowling_summary[["bowler"]]
    .drop_duplicates()
    .sort_values("bowler")
    .copy()
)
bowling_images["photo_url"] = ""

batting_images.to_csv(
    output_folder / "BattingPlayerImages.csv",
    index=False
)

bowling_images.to_csv(
    output_folder / "BowlingPlayerImages.csv",
    index=False
)

print("Batting players:", len(batting_images))
print("Bowling players:", len(bowling_images))
print("Both CSV files created successfully!")

Batting players: 123
Bowling players: 96
Both CSV files created successfully!
